# Fine-Tuning BERT for Text Classification 

## 1. Import all packages

In [2]:
from datasets import load_dataset

from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

import evaluate
import numpy as np
from transformers import DataCollatorWithPadding

/Users/pathmhajam/Git/Dartvolution/darland/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/pathmhajam/Git/Dartvolution/darland/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load the dataset

In [3]:
dataset_dict = load_dataset("shawhin/phishing-site-classification")

Generating test split: 100%|██████████| 450/450 [00:00<00:00, 235106.73 examples/s]


In [4]:
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 2100
    })
    validation: Dataset({
        features: ['text', 'labels'],
        num_rows: 450
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 450
    })
})

## 3. Load the model

In [5]:
# Load model directly
model_path = "google-bert/bert-base-uncased" # Has 110M parameters

tokenizer = AutoTokenizer.from_pretrained(model_path) # Take in arbitary text and convert to integer

id2label = {0: "Safe", 1: "Not Safe"}
label2id = {"Safe": 0, "Not Safe": 1}
model = AutoModelForSequenceClassification.from_pretrained(model_path, 
                                                           num_labels=2, 
                                                           id2label=id2label, 
                                                           label2id=label2id,)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 4. Set trainable parameters

We will freeze all the layers of the model except the last four layers. Unlike transfer learning, we will not freeze the entire model.

In [6]:
# print layers
for name, param in model.named_parameters():
   print(name, param.requires_grad)

bert.embeddings.word_embeddings.weight True
bert.embeddings.position_embeddings.weight True
bert.embeddings.token_type_embeddings.weight True
bert.embeddings.LayerNorm.weight True
bert.embeddings.LayerNorm.bias True
bert.encoder.layer.0.attention.self.query.weight True
bert.encoder.layer.0.attention.self.query.bias True
bert.encoder.layer.0.attention.self.key.weight True
bert.encoder.layer.0.attention.self.key.bias True
bert.encoder.layer.0.attention.self.value.weight True
bert.encoder.layer.0.attention.self.value.bias True
bert.encoder.layer.0.attention.output.dense.weight True
bert.encoder.layer.0.attention.output.dense.bias True
bert.encoder.layer.0.attention.output.LayerNorm.weight True
bert.encoder.layer.0.attention.output.LayerNorm.bias True
bert.encoder.layer.0.intermediate.dense.weight True
bert.encoder.layer.0.intermediate.dense.bias True
bert.encoder.layer.0.output.dense.weight True
bert.encoder.layer.0.output.dense.bias True
bert.encoder.layer.0.output.LayerNorm.weight True


In [7]:

# freeze base model parameters
for name, param in model.base_model.named_parameters():
    param.requires_grad = False

# unfreeze base model pooling layers
for name, param in model.base_model.named_parameters():
    if "pooler" in name:
        param.requires_grad = True

In [8]:

# print layers
for name, param in model.named_parameters():
   print(name, param.requires_grad)

bert.embeddings.word_embeddings.weight False
bert.embeddings.position_embeddings.weight False
bert.embeddings.token_type_embeddings.weight False
bert.embeddings.LayerNorm.weight False
bert.embeddings.LayerNorm.bias False
bert.encoder.layer.0.attention.self.query.weight False
bert.encoder.layer.0.attention.self.query.bias False
bert.encoder.layer.0.attention.self.key.weight False
bert.encoder.layer.0.attention.self.key.bias False
bert.encoder.layer.0.attention.self.value.weight False
bert.encoder.layer.0.attention.self.value.bias False
bert.encoder.layer.0.attention.output.dense.weight False
bert.encoder.layer.0.attention.output.dense.bias False
bert.encoder.layer.0.attention.output.LayerNorm.weight False
bert.encoder.layer.0.attention.output.LayerNorm.bias False
bert.encoder.layer.0.intermediate.dense.weight False
bert.encoder.layer.0.intermediate.dense.bias False
bert.encoder.layer.0.output.dense.weight False
bert.encoder.layer.0.output.dense.bias False
bert.encoder.layer.0.output.Lay

## 4. Preporcess Text

In [ ]:
# define text preprocessing
def preprocess_function(examples): # Takes a list of examples as input and returns a dictionary of tokenized examples
    return tokenizer(examples["text"], truncation=True)

# tokenize all datasetse
tokenized_data = dataset_dict.map(preprocess_function, batched=True) # batched=True means that the function is applied to the dataset in batches


Map: 100%|██████████| 450/450 [00:00<00:00, 38090.79 examples/s]


In [10]:
# create data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## 5. Evaluation

In [12]:
# load metrics
accuracy = evaluate.load("accuracy")
auc_score = evaluate.load("roc_auc")

def compute_metrics(eval_pred):
    # get predictions
    predictions, labels = eval_pred
    
    # apply softmax to get probabilities
    probabilities = np.exp(predictions) / np.exp(predictions).sum(-1, keepdims=True)
    # use probabilities of the positive class for ROC AUC
    positive_class_probs = probabilities[:, 1]
    # compute auc
    auc = np.round(auc_score.compute(prediction_scores=positive_class_probs, references=labels)['roc_auc'],3)
    
    # predict most probable class
    predicted_classes = np.argmax(predictions, axis=1)
    # compute accuracy
    acc = np.round(accuracy.compute(predictions=predicted_classes, references=labels)['accuracy'],3)
    
    return {"Accuracy": acc, "AUC": auc}

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## 6. Training parameters

In [13]:
# hyperparameters
lr = 2e-4
batch_size = 8
num_epochs = 10

training_args = TrainingArguments(
    output_dir="bert-phishing-classifier_teacher",
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

## 7. Fine-Tuning a Pre-trained Model

In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

/var/folders/21/zn8vzz352fvg29m2330b8zfh0000gn/T/ipykernel_6037/2732273287.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/Users/pathmhajam/Git/Dartvolution/darland/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,Auc
1,0.494400,0.419992,0.789000,0.913000
2,0.390400,0.355229,0.829000,0.932000
3,0.388400,0.315745,0.856000,0.939000
4,0.361800,0.438110,0.809000,0.942000
5,0.348400,0.327831,0.862000,0.946000
6,0.354000,0.303116,0.873000,0.948000
7,0.321700,0.291208,0.862000,0.949000
8,0.328500,0.296095,0.876000,0.949000
9,0.315700,0.289438,0.862000,0.950000
10,0.305800,0.297214,0.871000,0.951000


/Users/pathmhajam/Git/Dartvolution/darland/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/pathmhajam/Git/Dartvolution/darland/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/pathmhajam/Git/Dartvolution/darland/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/pathmhajam/Git/Dartvolution/darland/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings

TrainOutput(global_step=2630, training_loss=0.36092000460896656, metrics={'train_runtime': 334.3675, 'train_samples_per_second': 62.805, 'train_steps_per_second': 7.866, 'total_flos': 706603239165360.0, 'train_loss': 0.36092000460896656, 'epoch': 10.0})

## 8. Validation 

In [15]:
# apply model to validation dataset
predictions = trainer.predict(tokenized_data["validation"])

# Extract the logits and labels from the predictions object
logits = predictions.predictions
labels = predictions.label_ids

# Use your compute_metrics function
metrics = compute_metrics((logits, labels))
print(metrics)

/Users/pathmhajam/Git/Dartvolution/darland/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'Accuracy': np.float64(0.893), 'AUC': np.float64(0.944)}


In [18]:
from huggingface_hub import notebook_login
notebook_login()

## Pushing to Hugging Face Hub

In [19]:

# push model to hub
trainer.push_to_hub()


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (0 / 1)                :   0%|          | 14.2kB /  438MB, 4.43kB/s  

Processing Files (0 / 1)                :   0%|          |  582kB /  438MB,  162kB/s  


Processing Files (0 / 1)                :   0%|          | 1.15MB /  438MB,  274kB/s  

Processing Files (0 / 1)                :   0%|          | 1.72MB /  438MB,  374kB/s  
Processing Files (0 / 1)                :   1%|          | 2.86MB /  438MB,  595kB/s  
Processing Files (0 / 1)                :   1%|          | 5.13MB /  438MB, 1.03MB/s  
Processing Files (0 / 1)                :   2%|▏         | 8.54MB /  438MB, 1.64MB/s  
Processing Files (0 / 1)                :   3%|▎         | 11.4MB /  438MB, 2.11MB/s  
Processing Files (0 / 1)                :   3%|▎         | 14.8MB /  438MB, 2.64MB/s  
Processing Files (0 / 1)                :   4%|▍         | 18.2MB /  438MB, 3.14MB/s  
Processing Files (0 / 1)                : 

CommitInfo(commit_url='https://huggingface.co/Dart0050/bert-phishing-classifier_teacher/commit/1ad17799c477a80767d1ea2c962e1ef9c01fce7b', commit_message='End of training', commit_description='', oid='1ad17799c477a80767d1ea2c962e1ef9c01fce7b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Dart0050/bert-phishing-classifier_teacher', endpoint='https://huggingface.co', repo_type='model', repo_id='Dart0050/bert-phishing-classifier_teacher'), pr_revision=None, pr_num=None)

## 9. Run Inference on new examples

In [22]:
# First, check if CUDA is available
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("CUDA is not available. Using CPU instead.")
    
# Move your model to the appropriate device
model = model.to(device)

# Tokenize the input string
input_text = "grewbie.com"
inputs = tokenizer(input_text, return_tensors="pt").to(device)

# Perform inference
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)

# Map prediction to label
predicted_label = model.config.id2label[predictions.item()]
print(f"Predicted label: {predicted_label}")

CUDA is not available. Using CPU instead.
Predicted label: Safe
